# LSTM

用LSTM做月度销售预测。数据量比较小（36个月训练），主要是跑通pipeline。

做法：MinMaxScaler归一化，过去12个月预测下1个月，测试时递归预测。

In [ ]:
import sys
sys.path.append('..')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from src.data_loader import load_raw_data, get_monthly_sales, train_test_split_ts
from src.models.lstm_model import LSTMForecaster
from src.evaluate import evaluate_forecast, compare_models
from src.utils import plot_forecast, set_seed

set_seed(42)
plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

print(f'torch {torch.__version__}, cuda={torch.cuda.is_available()}')

In [ ]:
df = load_raw_data('../data/raw/train.csv')
monthly = get_monthly_sales(df)
train, test = train_test_split_ts(monthly, test_year=2018)
print(f'Train {len(train)}m, Test {len(test)}m')
print(f'去掉seq_len=12后有效样本: {len(train)-12}')

## 默认配置先跑一版

In [ ]:
lstm = LSTMForecaster(
    seq_len=12, hidden_size=64, num_layers=2,
    lr=0.001, epochs=150, batch_size=8,
)
lstm.fit(train)

In [ ]:
lstm.plot_loss()
plt.show()

loss收敛了但有波动，样本量少导致的。

In [ ]:
pred = lstm.predict(steps=len(test), test_index=test.index)
res = evaluate_forecast(test, pred, 'LSTM')
print(res)

In [ ]:
fig = plot_forecast(train, test, pred, 'LSTM (seq=12, h=64)')
plt.savefig('../results/figures/lstm_forecast.png', dpi=150, bbox_inches='tight')
plt.show()

## 试几组超参数

In [ ]:
# 手动试几个就好了，grid search在这个数据量上没意义
cfgs = [
    {'seq_len': 6, 'hidden_size': 32, 'epochs': 150},
    {'seq_len': 12, 'hidden_size': 64, 'epochs': 150},
    {'seq_len': 12, 'hidden_size': 128, 'epochs': 200},
]

all_res = []
for i, c in enumerate(cfgs):
    print(f'\n--- #{i+1}: seq={c["seq_len"]}, h={c["hidden_size"]} ---')
    set_seed(42)
    m = LSTMForecaster(
        seq_len=c['seq_len'], hidden_size=c['hidden_size'],
        num_layers=2, lr=0.001, epochs=c['epochs'], batch_size=8,
    )
    m.fit(train)
    p = m.predict(steps=len(test), test_index=test.index)
    r = evaluate_forecast(test, p, f"seq={c['seq_len']},h={c['hidden_size']}")
    all_res.append(r)
    print(f'  RMSE={r["RMSE"]}')

In [ ]:
compare_models(all_res)

加大hidden没用甚至更差了，瓶颈不在模型容量。

## 和之前的结果对比

In [ ]:
import os
if os.path.exists('../results/model_comparison.csv'):
    print(pd.read_csv('../results/model_comparison.csv').to_string(index=False))
else:
    print('先跑 python run_experiment.py')

## 总结

LSTM在这个任务上不如统计模型，主要原因：
- 训练数据太少，扣掉窗口只剩24个样本
- 递归预测误差会累积，后面几个月偏差变大
- 季节性pattern很规律，直接用去年同月的值就已经很准了

可以改进的方向：用daily数据（~1400点）、加外部特征（holiday等）、或者direct multi-output避免递归累积。

对这个数据集来说SARIMA已经够用了。